# 0.Library

In [20]:
# general
import pandas as pd
import importlib
import joblib


# Paths
import sys
sys.path.append('../') 
from features import data_utils as du
import paths_data as paths

# Reload
importlib.reload(du)
importlib.reload(paths)

<module 'paths_data' from '/home/smira/myproject/detection_AD_with_VR_data/src/notebooks/../paths_data.py'>

# 1. Paths and Constants

In [21]:
# Read models path
models_paths_folder = paths.MODELS_DIR_SOL_ONE
best_model = "SelectKBest_SVR_20"

df_name_input = "cleaned_features_20_young_patients.csv"
df_path = paths.STAGE_THREE_DATA_PREPROCESSING / df_name_input

# outputs name and paths
df_name_output = f"predicted_20_young_patients_by_{best_model}.csv"
df_path_output = paths.PREDICTS_SOLTUION_ONE

# 2. Load best model and DF input

In [22]:
# Load the best model
loaded_model = joblib.load(models_paths_folder / f"{best_model}.pkl")
df = pd.read_csv(df_path)

Target = False

X = df.copy()

if 'MoCA_Score' in df.columns:
    X = X.drop(columns=['MoCA_Score'])
    Target = True

X.head()

,Age,Help_Rating_out_of_5,Tutorial_total_reading_time,Tutorial_intensity_reading_time,Tutorial_total_count_hover,Tutorial_total_duration_hover,Tutorial_mean_duration_hover,Tutorial_max_duration_hover,Tutorial_total_count_press,Tutorial_median_duration_press,...,Memory_Yaw_std,Memory_Pitch_std,Memory_Roll_std,Memory_Yaw_range,Memory_Roll_range,Memory_dominant_hand_mean_speed,Memory_not_dominant_hand_mean_speed,Memory_dominant_hand_z_range,Memory_not_dominant_hand_y_range,Gender_Male
0,22.0,0,32.97,0.89,12,24.95,1.39,5.69,8,0.14,...,32.73,168.53,174.12,93.06,359.98,0.26,0.03,0.68,0.11,1
1,23.0,0,28.38,0.88,14,15.30,0.70,2.60,8,0.10,...,33.31,133.18,160.58,143.37,359.99,0.17,0.01,0.74,0.03,1
2,23.0,0,52.16,0.93,59,16.58,0.55,2.52,8,0.10,...,32.73,148.71,173.85,95.19,359.98,0.18,0.03,0.71,0.45,1
3,22.0,1,40.72,0.91,22,25.25,1.26,4.09,8,0.14,...,33.15,169.39,165.91,98.49,359.91,0.10,0.02,0.56,0.55,1
4,22.0,0,60.58,0.94,21,16.58,0.87,5.96,8,0.14,...,26.70,169.60,165.21,82.38,359.95,0.21,0.11,0.68,0.19,1


# 2. Predict MoCA score by best model

In [23]:
y_pred = loaded_model.predict(X)

In [24]:
y_pred

array([27.46725233, 27.57951694, 28.01306912, 27.05992514, 28.03651433,
       27.9082486 , 27.99688406, 27.56882485, 27.2993215 , 28.06438651,
       28.00146339, 27.50901274, 27.42220401, 27.47599139, 27.83084145,
       27.58130601, 27.90639753, 27.64790664, 27.68969851, 27.68235225])

In [25]:
df['Predicted_MoCA'] = y_pred
df['Predicted_MoCA'].head()

0    27.467252
1    27.579517
2    28.013069
3    27.059925
4    28.036514
Name: Predicted_MoCA, dtype: float64

# 3. Convert MoCa score to severity level

In [26]:
if Target :
    df['severity_level'] = df['MoCA_Score'].apply(du.severity_level_MoCA)
    
df['predicted_severity_level'] = df['Predicted_MoCA'].apply(du.severity_level_MoCA)

In [27]:
if Target :
    df[['MoCA_Score','severity_level','predicted_severity_level','Predicted_MoCA']] # uncomment this line when the Df has a MoCA score

else: df[['predicted_severity_level','Predicted_MoCA']]

# 4. Save DF

In [28]:
df.to_csv(df_path_output / df_name_output, index=False)